# MGMT298D: Science and Strategy of AI
## Week 1: Linear Regression & Regularization
### UCLA Anderson School of Management

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style('whitegrid')

## Load Data

In [ ]:
# Load H&M sales data
url = "https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/main/HMData.csv"
df = pd.read_csv(url)

print(f"Full dataset: {len(df)} rows, {df.shape[1]} columns")
print(f"\nAvailable products ({df['name'].nunique()}):")
for name in sorted(df['name'].unique()):
    count = len(df[df['name'] == name])
    print(f"  {name} ({count} rows)")

## Select a Product

In [ ]:
# Change this string to pick a different product
PRODUCT = "Vest top"

df_product = df[df['name'] == PRODUCT].reset_index(drop=True)
print(f"Product: {PRODUCT}")
print(f"Rows: {len(df_product)}")
print(f"\nSales range: {df_product['sales'].min()} – {df_product['sales'].max()}")
print(f"Price range: {df_product['price'].min():.4f} – {df_product['price'].max():.4f}")
df_product[['id', 'sales', 'price']].head(10)

## Explore: Sales vs Price

In [ ]:
# Scatter plot — is there a relationship between price and sales?
plt.figure(figsize=(8, 5))
plt.scatter(df_product['price'], df_product['sales'], alpha=0.4, s=30)
plt.xlabel('Price')
plt.ylabel('Sales')
plt.title(f'{PRODUCT}: Sales vs Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 1: OLS with Price Only

In [ ]:
# Start simple — can price alone predict sales?
features_v1 = ['price']

X = df_product[features_v1].values
y = df_product['sales'].values

# Time-based 80/20 split
split = int(0.8 * len(df_product))
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

scaler_v1 = StandardScaler()
X_train_s = scaler_v1.fit_transform(X_train)
X_test_s = scaler_v1.transform(X_test)

ols_v1 = LinearRegression()
ols_v1.fit(X_train_s, y_train)
pred_v1 = ols_v1.predict(X_test_s)

mae_v1 = mean_absolute_error(y_test, pred_v1)
r2_v1 = r2_score(y_test, pred_v1)

print(f"OLS (price only)")
print(f"  MAE:  {mae_v1:.2f}")
print(f"  R²:   {r2_v1:.4f}")

In [ ]:
# Predictions vs actuals — price-only model
plt.figure(figsize=(8, 5))
plt.scatter(y_test, pred_v1, alpha=0.5, s=40)
mn, mx = min(y_test.min(), pred_v1.min()), max(y_test.max(), pred_v1.max())
plt.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect')
plt.xlabel('Actual Sales')
plt.ylabel('Predicted Sales')
plt.title(f'OLS (Price Only) — MAE: {mae_v1:.0f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 2: Feature Engineering

In [ ]:
# Add lag features and moving average to capture time-series patterns
df_product['lag_1'] = df_product['sales'].shift(1)
df_product['lag_2'] = df_product['sales'].shift(2)
df_product['ma_3'] = df_product['sales'].rolling(3).mean().shift(1)
df_product['price_change'] = df_product['price'].diff()

df_product.fillna(0, inplace=True)

print("New features:")
df_product[['sales', 'price', 'lag_1', 'lag_2', 'ma_3', 'price_change']].describe().round(2)

## Step 3: OLS with Engineered Features

In [ ]:
# Now use price + all engineered features
features_v2 = ['price', 'lag_1', 'lag_2', 'ma_3', 'price_change']

X2 = df_product[features_v2].values
y2 = df_product['sales'].values

X2_train, X2_test = X2[:split], X2[split:]
y2_train, y2_test = y2[:split], y2[split:]

scaler_v2 = StandardScaler()
X2_train_s = scaler_v2.fit_transform(X2_train)
X2_test_s = scaler_v2.transform(X2_test)

ols_v2 = LinearRegression()
ols_v2.fit(X2_train_s, y2_train)
pred_v2 = ols_v2.predict(X2_test_s)

mae_v2 = mean_absolute_error(y2_test, pred_v2)
r2_v2 = r2_score(y2_test, pred_v2)

print(f"OLS (price + engineered features)")
print(f"  MAE:  {mae_v2:.2f}")
print(f"  R²:   {r2_v2:.4f}")
print(f"\nImprovement over price-only: MAE dropped by {mae_v1 - mae_v2:.2f}")

In [ ]:
# Side-by-side: price-only vs engineered features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, pred_v1, alpha=0.5, s=40)
axes[0].plot([mn, mx], [mn, mx], 'r--', lw=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Price Only — MAE: {mae_v1:.0f}')
axes[0].grid(True, alpha=0.3)

mn2, mx2 = min(y2_test.min(), pred_v2.min()), max(y2_test.max(), pred_v2.max())
axes[1].scatter(y2_test, pred_v2, alpha=0.5, s=40, color='green')
axes[1].plot([mn2, mx2], [mn2, mx2], 'r--', lw=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'Price + Engineered Features — MAE: {mae_v2:.0f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## OLS Coefficients

In [ ]:
# What did OLS learn from each feature?
coef_df = pd.DataFrame({
    'Feature': features_v2,
    'Coefficient': ols_v2.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(coef_df.to_string(index=False))

plt.figure(figsize=(8, 4))
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color='steelblue')
plt.xlabel('Coefficient Value (standardized)')
plt.title('OLS Feature Coefficients')
plt.tight_layout()
plt.show()

## Step 4: Lasso Regression (L1 Regularization)

In [ ]:
# Lasso adds an L1 penalty — it can shrink coefficients to exactly zero
lasso = Lasso(alpha=1.0, max_iter=10000)
lasso.fit(X2_train_s, y2_train)
pred_lasso = lasso.predict(X2_test_s)

mae_lasso = mean_absolute_error(y2_test, pred_lasso)
r2_lasso = r2_score(y2_test, pred_lasso)

print(f"Lasso (alpha=1.0)")
print(f"  MAE:  {mae_lasso:.2f}")
print(f"  R²:   {r2_lasso:.4f}")
print(f"\nCoefficients:")
for feat, coef in zip(features_v2, lasso.coef_):
    marker = "  ← zeroed out" if coef == 0 else ""
    print(f"  {feat:15s}: {coef:8.3f}{marker}")

## Lasso: Effect of Alpha

In [ ]:
# How does the regularization strength affect Lasso?
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
lasso_results = []

for a in alphas:
    m = Lasso(alpha=a, max_iter=10000)
    m.fit(X2_train_s, y2_train)
    p = m.predict(X2_test_s)
    lasso_results.append({
        'Alpha': a,
        'MAE': mean_absolute_error(y2_test, p),
        'Non-Zero Coefs': int(np.sum(m.coef_ != 0))
    })

lasso_df = pd.DataFrame(lasso_results)
print(lasso_df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.semilogx(lasso_df['Alpha'], lasso_df['MAE'], 'o-', lw=2, ms=8)
ax1.set_xlabel('Alpha')
ax1.set_ylabel('MAE')
ax1.set_title('Lasso: Alpha vs Error')
ax1.grid(True, alpha=0.3)

ax2.semilogx(lasso_df['Alpha'], lasso_df['Non-Zero Coefs'], 's-', lw=2, ms=8, color='orange')
ax2.set_xlabel('Alpha')
ax2.set_ylabel('Non-Zero Coefficients')
ax2.set_title('Lasso: Alpha vs Sparsity')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Ridge Regression (L2 Regularization)

In [ ]:
# Ridge adds an L2 penalty — shrinks coefficients but never to zero
ridge = Ridge(alpha=1.0)
ridge.fit(X2_train_s, y2_train)
pred_ridge = ridge.predict(X2_test_s)

mae_ridge = mean_absolute_error(y2_test, pred_ridge)
r2_ridge = r2_score(y2_test, pred_ridge)

print(f"Ridge (alpha=1.0)")
print(f"  MAE:  {mae_ridge:.2f}")
print(f"  R²:   {r2_ridge:.4f}")
print(f"\nCoefficients:")
for feat, coef in zip(features_v2, ridge.coef_):
    print(f"  {feat:15s}: {coef:8.3f}")

## Ridge: Effect of Alpha

In [ ]:
# Same alpha sweep for Ridge
ridge_results = []

for a in alphas:
    m = Ridge(alpha=a)
    m.fit(X2_train_s, y2_train)
    p = m.predict(X2_test_s)
    ridge_results.append({'Alpha': a, 'MAE': mean_absolute_error(y2_test, p)})

ridge_df = pd.DataFrame(ridge_results)
print(ridge_df.to_string(index=False))

plt.figure(figsize=(8, 5))
plt.semilogx(ridge_df['Alpha'], ridge_df['MAE'], 'o-', lw=2, ms=8, color='green')
plt.xlabel('Alpha')
plt.ylabel('MAE')
plt.title('Ridge: Alpha vs Error')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 6: ElasticNet (L1 + L2)

In [ ]:
# ElasticNet blends Lasso and Ridge penalties
elastic = ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
elastic.fit(X2_train_s, y2_train)
pred_elastic = elastic.predict(X2_test_s)

mae_elastic = mean_absolute_error(y2_test, pred_elastic)
r2_elastic = r2_score(y2_test, pred_elastic)

print(f"ElasticNet (alpha=1.0, l1_ratio=0.5)")
print(f"  MAE:  {mae_elastic:.2f}")
print(f"  R²:   {r2_elastic:.4f}")

## Final Model Comparison

In [ ]:
# Collect all results
all_results = pd.DataFrame([
    {'Model': 'OLS (price only)',      'MAE': mae_v1,      'R²': r2_v1},
    {'Model': 'OLS (all features)',    'MAE': mae_v2,      'R²': r2_v2},
    {'Model': 'Lasso (alpha=1.0)',     'MAE': mae_lasso,   'R²': r2_lasso},
    {'Model': 'Ridge (alpha=1.0)',     'MAE': mae_ridge,   'R²': r2_ridge},
    {'Model': 'ElasticNet (alpha=1.0)','MAE': mae_elastic, 'R²': r2_elastic},
])

print(all_results.to_string(index=False))

In [ ]:
# Bar chart of MAE across all models
plt.figure(figsize=(10, 5))
colors = ['#aaa', '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
bars = plt.bar(all_results['Model'], all_results['MAE'], color=colors, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, all_results['MAE']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{val:.0f}',
             ha='center', va='bottom', fontsize=10)
plt.ylabel('MAE')
plt.title('Model Comparison: Mean Absolute Error')
plt.xticks(rotation=20, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Coefficient Comparison

In [ ]:
# How do regularized models reshape the coefficients?
coef_compare = pd.DataFrame({
    'Feature': features_v2,
    'OLS': ols_v2.coef_,
    'Lasso': lasso.coef_,
    'Ridge': ridge.coef_,
    'ElasticNet': elastic.coef_
})

print(coef_compare.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(features_v2))
w = 0.2
ax.bar(x - 1.5*w, ols_v2.coef_,    w, label='OLS')
ax.bar(x - 0.5*w, lasso.coef_,     w, label='Lasso')
ax.bar(x + 0.5*w, ridge.coef_,     w, label='Ridge')
ax.bar(x + 1.5*w, elastic.coef_,   w, label='ElasticNet')
ax.set_xticks(x)
ax.set_xticklabels(features_v2)
ax.set_ylabel('Coefficient')
ax.set_title('Coefficient Comparison Across Models')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Residual Analysis

In [ ]:
# Pick the best model and check residual distribution
best_name = all_results.loc[all_results['MAE'].idxmin(), 'Model']
best_preds = {'OLS (price only)': pred_v1, 'OLS (all features)': pred_v2,
              'Lasso (alpha=1.0)': pred_lasso, 'Ridge (alpha=1.0)': pred_ridge,
              'ElasticNet (alpha=1.0)': pred_elastic}
residuals = y2_test - best_preds[best_name]

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=25, edgecolor='black', alpha=0.7)
plt.xlabel('Residual (Actual − Predicted)')
plt.ylabel('Frequency')
plt.title(f'Residual Distribution — {best_name}')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()